# Pipeline: damage_classification (modelo 2 - tipo de dano + ubicacion)

Procesa varios datasets de origen, cada uno en su propia seccion independiente, porque cada dataset puede tener un formato de XML y una logica de parseo distinta. Todas escriben en la MISMA carpeta final, fusionando el mismo `classes.txt` / `data.yaml` (no se pisan entre si).

## Configuracion comun

In [13]:
import sys
import os
from pathlib import PurePosixPath

current = os.getcwd()
etl_dir = None
for _ in range(5):
    candidate = os.path.join(current, "etl")
    if os.path.isdir(candidate):
        etl_dir = candidate
        break
    current = os.path.dirname(current)

if etl_dir is None:
    raise RuntimeError("No encontre la carpeta 'etl'. Revisar la estructura del proyecto.")

if etl_dir not in sys.path:
    sys.path.append(etl_dir)

print(f"etl encontrado en: {etl_dir}")

from pathlib import PurePosixPath
from extract.extract_annotations import list_xml_keys, download_xml
from load.load_to_s3 import copy_image, upload_bytes, download_text_if_exists

BUCKET = "humanoid-damage-cnn-834257582282-us-east-1-an"
PROCESSED_PREFIX = "processed/damage_classification/"
DRY_RUN = True  # cambiar a False cuando quieras subir de verdad

etl encontrado en: /home/sagemaker-user/humanoid-damage-cnn/etl


## Funciones compartidas de fusion y carga

Esto es lo unico que se repite igual para cualquier dataset, una vez que ya tenes la lista de objetos (tipo de dano + caja en pixeles). Se deja acá (no en un .py aparte) para que lo puedas ver y ajustar facil si hace falta.

In [7]:
def load_existing_classes(bucket, processed_prefix):
    """Lee el classes.txt actual (si existe) para no perder los IDs ya asignados."""
    content = download_text_if_exists(bucket, f"{processed_prefix}classes.txt")
    if content is None:
        return []
    return [line.strip() for line in content.splitlines() if line.strip()]


def merge_classes(existing_classes, new_damage_types):
    """Agrega al final los tipos de dano nuevos, sin tocar el orden/ID de los existentes."""
    merged = list(existing_classes)
    for damage_type in sorted(new_damage_types):
        if damage_type not in merged:
            merged.append(damage_type)
    return merged


def to_yolo_line(obj, class_to_id, width, height):
    """Convierte un objeto (dano con bndbox en pixeles) a una linea YOLO normalizada."""
    class_id = class_to_id[obj["damage_type"]]
    x_center = ((obj["xmin"] + obj["xmax"]) / 2) / width
    y_center = ((obj["ymin"] + obj["ymax"]) / 2) / height
    box_width = (obj["xmax"] - obj["xmin"]) / width
    box_height = (obj["ymax"] - obj["ymin"]) / height
    return f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}"


def run_classification_etl(bucket, raw_annotations_prefix, raw_images_prefix, processed_prefix, parse_fn, dry_run=True):
    """Corre extract -> transform -> load para un dataset, fusionando con lo que ya haya en processed_prefix.
    parse_fn(xml_bytes) debe devolver: {"image_basename": str, "width": int, "height": int,
        "objects": [{"damage_type": str, "xmin": float, "ymin": float, "xmax": float, "ymax": float}, ...]}"""

    # --- Extract ---
    xml_keys = list_xml_keys(bucket, raw_annotations_prefix)
    print(f"Encontrados {len(xml_keys)} archivos XML en {raw_annotations_prefix}")

    # --- Transform ---
    parsed = []
    for key in xml_keys:
        try:
            xml_bytes = download_xml(bucket, key)
            annotation = parse_fn(xml_bytes)
        except Exception as exc:
            print(f"[ERROR] {key}: {exc}")
            continue
        if not annotation.get("image_basename"):
            print(f"[ERROR] {key}: no se pudo determinar el nombre de la imagen")
            continue
        if not annotation.get("objects"):
            continue  # sin daños anotados, no aporta a este dataset
        parsed.append(annotation)

    print(f"Anotaciones con objetos: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = f"{raw_images_prefix}{basename}"

        dest_image_key = f"{processed_prefix}images/{basename}"
        copy_image(bucket, source_key, dest_image_key, dry_run=False)

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 50 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print("Listo.")
    return {"parsed": parsed, "all_classes": all_classes}

---
## Dataset `raw/IDEA/damages/`

Formato: un XML por imagen, tipo Pascal VOC (`<object><name damage="...">` + `<bndbox>`). El parser es especifico de este dataset, por eso vive acá y no en el bloque compartido de arriba.

In [3]:
import xml.etree.ElementTree as ET

def parse_dataset_1(xml_bytes):
    root = ET.fromstring(xml_bytes)

    filename_tag = root.findtext("filename", default="").strip()
    image_basename = PurePosixPath(filename_tag).name if filename_tag else None

    size_elem = root.find("size")
    width = int(size_elem.findtext("width")) if size_elem is not None else None
    height = int(size_elem.findtext("height")) if size_elem is not None else None

    objects = []
    for obj in root.findall("object"):
        name_elem = obj.find("name")
        if name_elem is None:
            continue
        damage_type = name_elem.attrib.get("damage") or (name_elem.text or "").strip()
        if not damage_type:
            continue
        bndbox = obj.find("bndbox")
        if bndbox is None:
            continue
        objects.append({
            "damage_type": damage_type.strip(),
            "xmin": float(bndbox.findtext("xmin")),
            "ymin": float(bndbox.findtext("ymin")),
            "xmax": float(bndbox.findtext("xmax")),
            "ymax": float(bndbox.findtext("ymax")),
        })

    return {"image_basename": image_basename, "width": width, "height": height, "objects": objects}

In [4]:
RAW_ANNOTATIONS_PREFIX_1 = "raw/IDEA/damages/annotations/"
RAW_IMAGES_PREFIX_1 = "raw/IDEA/damages/images/"

resultado_1 = run_classification_etl(
    bucket=BUCKET,
    raw_annotations_prefix=RAW_ANNOTATIONS_PREFIX_1,
    raw_images_prefix=RAW_IMAGES_PREFIX_1,
    processed_prefix=PROCESSED_PREFIX,
    parse_fn=parse_dataset_1,
    dry_run=DRY_RUN,
)

Encontrados 1727 archivos XML en raw/IDEA/damages/annotations/
Anotaciones con objetos: 1727
Tipos de dano nuevos agregados a classes.txt: ['Clay block damage', 'Concrete spalling', 'Corrosion/delamination', 'Crack', 'Crushing crack', 'Debris', 'Detachment crack', 'Detachment of tympanum', 'Flexural crack', 'Flexural/rocking crack', 'Impact damages', 'Large displacement', 'Leaching', 'Masonry spalling', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar buckling', 'Rebar deformation', 'Reinforcement oxidized/corroded', 'Shear crack', 'Sliding shear crack', 'Stirrups break']
Total de clases acumuladas: 23 -> ['Clay block damage', 'Concrete spalling', 'Corrosion/delamination', 'Crack', 'Crushing crack', 'Debris', 'Detachment crack', 'Detachment of tympanum', 'Flexural crack', 'Flexural/rocking crack', 'Impact damages', 'Large displacement', 'Leaching', 'Masonry spalling', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar bu

---
## Dataset `raw/Structural Defects 2.v1i.coco/damages/`

Formato: **COCO** — un solo JSON con `categories` (tipos de dano), `images` (metadata) y `annotations` (cajas, cada una con `image_id` apuntando a una imagen). El `bbox` viene como `[x_min, y_min, ancho, alto]` en pixeles (distinto al `xmin/ymin/xmax/ymax` de la Cajita 1), asi que hay que convertirlo.

Como es un solo archivo (no uno por imagen), la logica de extract/transform es distinta a la Cajita 1, pero el Load final usa las MISMAS funciones compartidas (`load_existing_classes`, `merge_classes`, `to_yolo_line`, `copy_image`, `upload_bytes`) para terminar en la misma carpeta.

In [11]:
import json

# Traduccion de los nombres de categoria (vienen en italiano en este dataset)
ITALIAN_TO_ENGLISH = {
    "Armatura corrosa": "Corroded reinforcement",
    "Corrosione": "Corrosion",
    "Distacco del copriferro": "Concrete cover detachment",
    "Fessura diagonale": "Diagonal crack",
}

def run_classification_etl_coco(bucket, annotation_key, raw_images_prefix, processed_prefix, dry_run=True):
    """Version de run_classification_etl para un dataset en formato COCO (un solo JSON para todas las imagenes)."""

    # --- Extract ---
    raw_bytes = download_xml(bucket, annotation_key)  # funciona igual, solo descarga bytes
    coco = json.loads(raw_bytes)
    print(f"JSON descargado: {len(coco['images'])} imagenes, {len(coco['annotations'])} anotaciones")

    # --- Transform ---
    category_id_to_name = {}
    for c in coco["categories"]:
        nombre_original = c["name"]
        nombre_en_ingles = ITALIAN_TO_ENGLISH.get(nombre_original, nombre_original)
        if nombre_original not in ITALIAN_TO_ENGLISH and nombre_original != "concrete-5ILF":
            print(f"[AVISO] '{nombre_original}' no esta en el diccionario de traduccion, se deja tal cual.")
        category_id_to_name[c["id"]] = nombre_en_ingles
    images_by_id = {img["id"]: img for img in coco["images"]}

    objects_by_image_id = {}
    for ann in coco["annotations"]:
        image_id = ann["image_id"]
        x_min, y_min, box_w, box_h = ann["bbox"]
        obj = {
            "damage_type": category_id_to_name[ann["category_id"]],
            "xmin": float(x_min),
            "ymin": float(y_min),
            "xmax": float(x_min) + float(box_w),
            "ymax": float(y_min) + float(box_h),
        }
        objects_by_image_id.setdefault(image_id, []).append(obj)

    parsed = []
    for image_id, objects in objects_by_image_id.items():
        img = images_by_id.get(image_id)
        if img is None:
            print(f"[ERROR] image_id {image_id} no encontrado en 'images'")
            continue
        parsed.append({
            "image_basename": img["file_name"],
            "width": img["width"],
            "height": img["height"],
            "objects": objects,
        })

    print(f"Imagenes con objetos: {len(parsed)}")

    new_damage_types = {obj["damage_type"] for a in parsed for obj in a["objects"]}
    existing_classes = load_existing_classes(bucket, processed_prefix)
    all_classes = merge_classes(existing_classes, new_damage_types)
    class_to_id = {name: i for i, name in enumerate(all_classes)}

    nuevas = sorted(new_damage_types - set(existing_classes))
    if nuevas:
        print(f"Tipos de dano nuevos agregados a classes.txt: {nuevas}")
    print(f"Total de clases acumuladas: {len(all_classes)} -> {all_classes}")

    if dry_run:
        print("\n[DRY RUN] No se subio nada.")
        return {"parsed": parsed, "all_classes": all_classes}

    # --- Load (identico a la Cajita 1) ---
    upload_bytes(bucket, f"{processed_prefix}classes.txt", ("\n".join(all_classes) + "\n").encode("utf-8"), dry_run=False)

    data_yaml = f"train: images\nnc: {len(all_classes)}\nnames: {all_classes}\n"
    upload_bytes(bucket, f"{processed_prefix}data.yaml", data_yaml.encode("utf-8"), dry_run=False)

    for i, annotation in enumerate(parsed, start=1):
        basename = annotation["image_basename"]
        source_key = f"{raw_images_prefix}{basename}"

        dest_image_key = f"{processed_prefix}images/{basename}"
        copy_image(bucket, source_key, dest_image_key, dry_run=False)

        label_lines = [to_yolo_line(obj, class_to_id, annotation["width"], annotation["height"]) for obj in annotation["objects"]]
        label_key = f"{processed_prefix}labels/{PurePosixPath(basename).stem}.txt"
        upload_bytes(bucket, label_key, ("\n".join(label_lines) + "\n").encode("utf-8"), dry_run=False)

        if i % 200 == 0 or i == len(parsed):
            print(f"Procesadas {i}/{len(parsed)} imagenes...")

    print("Listo.")
    return {"parsed": parsed, "all_classes": all_classes}

In [12]:
ANNOTATION_KEY_2 = "raw/Structural Defects 2.v1i.coco/damages/annotations/_annotations.coco.json"
RAW_IMAGES_PREFIX_2 = "raw/Structural Defects 2.v1i.coco/damages/images/"

resultado_2 = run_classification_etl_coco(
    bucket=BUCKET,
    annotation_key=ANNOTATION_KEY_2,
    raw_images_prefix=RAW_IMAGES_PREFIX_2,
    processed_prefix=PROCESSED_PREFIX,
    dry_run=DRY_RUN,
)

JSON descargado: 6762 imagenes, 10689 anotaciones
Imagenes con objetos: 4726
Tipos de dano nuevos agregados a classes.txt: ['Concrete cover detachment', 'Corrosion', 'Diagonal crack']
Total de clases acumuladas: 26 -> ['Clay block damage', 'Concrete spalling', 'Corrosion/delamination', 'Crack', 'Crushing crack', 'Debris', 'Detachment crack', 'Detachment of tympanum', 'Flexural crack', 'Flexural/rocking crack', 'Impact damages', 'Large displacement', 'Leaching', 'Masonry spalling', 'Non structural damage', 'Out-of-plane mechanism', 'Partial/full collapse', 'Rebar buckling', 'Rebar deformation', 'Reinforcement oxidized/corroded', 'Shear crack', 'Sliding shear crack', 'Stirrups break', 'Concrete cover detachment', 'Corrosion', 'Diagonal crack']
Procesadas 200/4726 imagenes...
Procesadas 400/4726 imagenes...
Procesadas 600/4726 imagenes...
Procesadas 800/4726 imagenes...
Procesadas 1000/4726 imagenes...
Procesadas 1200/4726 imagenes...
Procesadas 1400/4726 imagenes...
Procesadas 1600/4726 